# Module 5: Agent Frameworks
# Topics 41, 42 & 43: Document Loaders, Text Splitters & Embeddings

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Most Important for RAG)
>
> **Interview Frequency:** Extremely High
>
> **Prerequisites:**
> - Models ✅
> - LCEL ✅
> - Memory ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is a Document Loader?
- Why do we split documents?
- What are Embeddings?
- Complete RAG ingestion pipeline
- Types of Document Loaders
- Text Splitter strategies
- Chunk Size & Chunk Overlap
- Embedding Models
- Best Practices
- Interview Questions

---

# 1. The RAG Ingestion Pipeline

Before a chatbot can answer questions from your documents, those documents must be prepared.

```text
PDF

↓

Document Loader

↓

Document

↓

Text Splitter

↓

Chunks

↓

Embedding Model

↓

Vectors

↓

Vector Database
```

Everything before the Vector Database is called the **Ingestion Pipeline**.

---

# Part 1: Document Loaders

# 2. What is a Document Loader?

A **Document Loader** reads data from different sources and converts it into LangChain `Document` objects.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **A Document Loader is a LangChain component responsible for reading data from external sources and converting it into standardized `Document` objects for downstream processing.**

---

# 3. Why Do We Need Document Loaders?

Different data sources have different formats.

Examples:

- PDF
- DOCX
- TXT
- HTML
- CSV
- Database
- Web Page
- API

Instead of writing custom code for each source, LangChain provides reusable loaders.

---

# 4. Document Loader Architecture

```text
PDF

↓

PDF Loader

↓

Document

↓

Text Splitter
```

---

# 5. Common Document Loaders

| Loader | Reads |
|---------|-------|
| PyPDFLoader | PDF |
| TextLoader | TXT |
| CSVLoader | CSV |
| Docx2txtLoader | Word |
| WebBaseLoader | Website |
| DirectoryLoader | Folder |
| UnstructuredLoader | Multiple formats |

---

# 6. Example - PDF Loader

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("employee_handbook.pdf")

documents = loader.load()

print(documents[0].page_content)
```

Output

```
Employee Leave Policy...
```

---

# 7. What is a Document?

Every loader returns a `Document`.

```python
Document(
    page_content="...",
    metadata={
        "source":"employee_handbook.pdf",
        "page":1
    }
)
```

---

# 8. Why Metadata?

Metadata helps during retrieval.

Example

```python
{
    "page":15,
    "source":"policy.pdf"
}
```

Later you can cite:

```
Answer found on Page 15.
```

---

# Part 2: Text Splitters

# 9. Why Split Documents?

Suppose you have

```
Employee Handbook

300 Pages
```

LLMs cannot process such large documents efficiently.

Instead

```
Document

↓

Chunks
```

---

# Interview Definition ⭐⭐⭐⭐⭐

> **A Text Splitter divides large documents into smaller chunks so they fit within the model's context window and improve retrieval quality.**

---

# 10. Splitting Architecture

```text
Document

↓

Text Splitter

↓

Chunk 1

Chunk 2

Chunk 3

Chunk 4
```

---

# 11. RecursiveCharacterTextSplitter

The most commonly used splitter.

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100

)
```

---

# 12. Splitting Example

Original

```
3000 Characters
```

↓

Split

```
Chunk 1

500 Characters

↓

Chunk 2

500 Characters

↓

Chunk 3

500 Characters
```

---

# 13. Chunk Size

Controls how large each chunk is.

Example

```python
chunk_size=500
```

Larger chunk

```
More Context

Higher Cost
```

Smaller chunk

```
Less Cost

Less Context
```

---

# 14. Chunk Overlap

Suppose sentence

```
LangChain is an
AI framework
for building
applications.
```

Without overlap

Chunk 1

```
LangChain is an AI
```

Chunk 2

```
framework for building...
```

Meaning lost.

---

With overlap

Chunk 1

```
LangChain is an AI framework
```

Chunk 2

```
AI framework for building...
```

The repeated text preserves context across chunk boundaries.

---

# 15. Choosing Chunk Size

General guidelines

| Use Case | Chunk Size |
|----------|-----------:|
| FAQ | 200–400 |
| Documentation | 500–800 |
| Research Papers | 800–1200 |
| Books | 1000–1500 |

---

# Part 3: Embeddings

# 16. What are Embeddings?

LLMs understand text.

Vector databases understand numbers.

Embeddings convert text into numerical vectors.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **An Embedding is a dense numerical vector that captures the semantic meaning of text, enabling similarity search and retrieval.**

---

# 17. Embedding Architecture

```text
Text

↓

Embedding Model

↓

Vector

↓

Vector Database
```

---

# 18. Example

Text

```
Artificial Intelligence
```

Embedding

```text
[0.21,
0.44,
-0.18,
0.79,
...]
```

Hundreds or thousands of floating-point values represent the meaning.

---

# 19. Similar Meaning = Similar Vectors

```
Car

↓

Vector A
```

```
Automobile

↓

Vector B
```

Cosine Similarity

```
0.97
```

Very similar.

---

```
Car

↓

Vector A
```

```
Banana

↓

Vector C
```

Similarity

```
0.08
```

Very different.

---

# 20. Embedding Workflow

```text
Chunk

↓

Embedding Model

↓

Vector

↓

Store

↓

Vector Database
```

Later

```text
Question

↓

Embedding

↓

Similarity Search

↓

Relevant Chunks
```

---

# 21. Common Embedding Models

| Provider | Model |
|----------|-------|
| OpenAI | text-embedding-3-small |
| OpenAI | text-embedding-3-large |
| Hugging Face | BAAI/bge-small-en |
| Hugging Face | BAAI/bge-base-en |
| Hugging Face | all-MiniLM-L6-v2 |
| Google | Gemini Embeddings |

---

# 22. Example - OpenAI Embeddings

```python
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector = embeddings.embed_query(
    "What is LangChain?"
)

print(len(vector))
```

Output

```
1536
```

(Embedding dimension depends on the selected model.)

---

# 23. Complete Ingestion Pipeline

```text
PDF

↓

PyPDFLoader

↓

Document

↓

RecursiveCharacterTextSplitter

↓

Chunks

↓

OpenAI Embeddings

↓

Vectors

↓

Vector Database
```

This pipeline is used in almost every RAG application.

---

# 24. Real Enterprise Example

HR Policy Chatbot

```text
Employee Handbook.pdf

↓

PyPDFLoader

↓

Chunks

↓

Embeddings

↓

Qdrant

↓

Question

↓

Similarity Search

↓

Relevant Chunks

↓

LLM

↓

Answer
```

---

# 25. Best Practices

### Document Loaders

✅ Preserve metadata.

✅ Remove duplicate documents.

✅ Validate document encoding.

---

### Text Splitters

✅ Start with `RecursiveCharacterTextSplitter`.

✅ Choose chunk sizes based on document type.

✅ Use chunk overlap (typically 10–20% of chunk size).

---

### Embeddings

✅ Use the same embedding model for indexing and querying.

✅ Normalize and clean text before embedding when appropriate.

✅ Re-embed documents if you change the embedding model.

---

# 26. Common Mistakes

❌ Embedding the entire PDF without splitting.

❌ Setting chunk size too large.

❌ Setting chunk overlap to zero for long documents.

❌ Using different embedding models for indexing and querying.

❌ Ignoring document metadata.

---

# 27. Interview Questions

## Q1. What is a Document Loader?

**Answer:**

A Document Loader reads data from external sources such as PDFs, Word files, websites, or databases and converts it into LangChain `Document` objects.

---

## Q2. Why do we split documents?

**Answer:**

Large documents exceed the model's context window and reduce retrieval accuracy. Splitting them into smaller chunks improves indexing, retrieval, and LLM performance.

---

## Q3. Why is chunk overlap important?

**Answer:**

Chunk overlap preserves context across chunk boundaries, preventing important information from being split into isolated chunks.

---

## Q4. What are Embeddings?

**Answer:**

Embeddings are dense numerical vectors representing the semantic meaning of text. They enable similarity search in vector databases.

---

## Q5. Why must the same embedding model be used for indexing and querying?

**Answer:**

Different embedding models generate vectors in different semantic spaces. Using different models for indexing and querying can significantly reduce retrieval accuracy.

---

## Q6. What metadata is stored in a Document?

**Answer:**

Typical metadata includes the source file, page number, document ID, author, creation date, or any other information useful during retrieval and citation.

---

# 28. Quick Revision

| Component | Purpose |
|-----------|----------|
| Document Loader | Reads external data |
| Document | Standard LangChain object |
| Text Splitter | Creates chunks |
| Chunk Size | Size of each chunk |
| Chunk Overlap | Preserves context |
| Embedding | Converts text into vectors |

---

# Interview Cheat Sheet

```text
PDF

↓

Document Loader

↓

Document

↓

Text Splitter

↓

Chunks

↓

Embedding Model

↓

Vectors

↓

Vector Database

Key Concepts

Document

Metadata

Chunk Size

Chunk Overlap

Embedding

Semantic Search
```

---

# 29. Real Interview Scenario

**Question:**

> Your RAG chatbot gives poor answers even though all documents are indexed. What would you investigate first?

**Answer:**

I would verify the ingestion pipeline:
1. Check whether documents were loaded correctly.
2. Ensure chunk size and overlap are appropriate.
3. Confirm the same embedding model is used for both indexing and querying.
4. Validate that embeddings were successfully stored in the vector database.
5. Inspect the retrieved chunks to ensure relevant context is being returned before the LLM generates an answer.

---

# 30-Second Interview Answer

> **The RAG ingestion pipeline begins by loading documents using Document Loaders, converting them into standardized `Document` objects with metadata. These documents are then divided into manageable chunks using a Text Splitter. Each chunk is transformed into a semantic vector using an Embedding Model and stored in a Vector Database. During retrieval, the user's query is embedded using the same model, and the most semantically similar chunks are returned to the LLM to generate an accurate, context-aware response.**

---

# Key Takeaway

> **A successful RAG system depends more on a high-quality ingestion pipeline than on the LLM itself. Proper document loading, intelligent chunking, and consistent embeddings form the foundation of accurate semantic retrieval.**